# Phase 4 YOLO Baseline

Official YOLOv8n baseline on WBF labels and the Phase 3 fixed split. This run is intentionally conservative for Kaggle P100.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

print('INPUT DIRS')
for p in Path('/kaggle/input').iterdir():
    print(p)

gpu_name = subprocess.run(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    capture_output=True,
    text=True,
).stdout.strip()
print('GPU:', gpu_name or '<none>')

## 1. Install Runtime Dependencies

In [ ]:
# P100 has compute capability sm_60; Kaggle default torch can be incompatible.
if 'P100' in gpu_name:
    !pip install --no-cache-dir torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121

!pip install --no-cache-dir ultralytics ensemble-boxes tqdm

## 2. Clone Latest Project Code

In [ ]:
%cd /kaggle/working
repo = Path('cxr-detectbench')
if repo.exists():
    shutil.rmtree(repo)
!git clone https://github.com/kinjazA/cxr-detectbench.git
%cd /kaggle/working/cxr-detectbench
!git rev-parse --short HEAD

## 3. Prepare Raw Inputs and PNG Symlink

In [ ]:
!mkdir -p data/raw data/processed

train_csv_candidates = sorted(Path('/kaggle/input').rglob('train.csv'))
if not train_csv_candidates:
    raise FileNotFoundError('No train.csv found under /kaggle/input')
train_csv = train_csv_candidates[0]
print('train_csv:', train_csv)
shutil.copy2(train_csv, 'data/raw/train.csv')

train_meta = Path('/kaggle/input/datasets/corochann/vinbigdata-chest-xray-original-png/train_meta.csv')
train_png_dir = Path('/kaggle/input/datasets/corochann/vinbigdata-chest-xray-original-png/train')
if not train_meta.exists():
    raise FileNotFoundError(f'Missing confirmed train_meta.csv: {train_meta}')
if not train_png_dir.is_dir():
    raise FileNotFoundError(f'Missing confirmed PNG train dir: {train_png_dir}')
png_count = len(list(train_png_dir.glob('*.png')))
print('train_meta:', train_meta)
print('train_png_dir:', train_png_dir, 'png_count=', png_count)
if png_count != 15000:
    raise RuntimeError(f'Expected 15000 train PNG files, got {png_count}')
shutil.copy2(train_meta, 'data/raw/images.csv')

images_link = Path('data/processed/images_png')
if images_link.is_symlink() or images_link.is_file():
    images_link.unlink()
elif images_link.exists():
    shutil.rmtree(images_link)
images_link.symlink_to(train_png_dir, target_is_directory=True)
print('images_link:', images_link, '->', os.path.realpath(images_link))
print('linked_png_count:', len(list(images_link.glob('*.png'))))

## 4. Build WBF COCO and Official YOLO Dataset

In [ ]:
!python scripts/label_fusion.py \
    --train_csv data/raw/train.csv \
    --images_csv data/raw/images.csv \
    --output_dir data/processed/labels_coco \
    --iou_thr 0.5

!python scripts/prepare_yolo_dataset.py \
    --coco_json data/processed/labels_coco/wbf/annotations.json \
    --images_dir data/processed/images_png \
    --splits_dir data/splits \
    --output_dir data/processed/yolo_wbf_phase3 \
    --overwrite

!cat data/processed/yolo_wbf_phase3/dataset_summary.csv

## 5. Train YOLOv8n Baseline

In [ ]:
!python scripts/train_yolo_baseline.py \
    --data data/processed/yolo_wbf_phase3/data.yaml \
    --model yolov8n.pt \
    --epochs 50 \
    --imgsz 640 \
    --batch 16 \
    --workers 4 \
    --seed 42 \
    --project /kaggle/working/yolo_baseline_runs \
    --name yolov8n_wbf_phase3_img640_ep50 \
    --summary_dir /kaggle/working/yolo_baseline_summary \
    --dataset_summary data/processed/yolo_wbf_phase3/dataset_summary.csv \
    --exist_ok \
    --include_last

## 6. Keep Output Compact

In [ ]:
print('=== yolo_baseline_summary ===')
!find /kaggle/working/yolo_baseline_summary -maxdepth 2 -type f -printf '%P %s bytes\n'

# Keep only the compact summary. The cloned repo and generated YOLO dataset are reproducible.
for path in [
    Path('/kaggle/working/cxr-detectbench'),
    Path('/kaggle/working/yolo_baseline_runs'),
    Path('/kaggle/working/yolov8n.pt'),
    Path('/kaggle/working/yolo26n.pt'),
]:
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)
print('Cleaned regenerated files from /kaggle/working')